# Aula 5, Qdrant para produção

Notebook executável que acompanha a aula [05-qdrant-producao.md](../../lessons/modulo-09-rag/05-qdrant-producao.md).

## O que vamos fazer aqui

Em produção, importam recursos como a filtragem por metadados. Vamos demonstrar esse recurso,
buscando só dentro de uma disciplina, com o nosso banco mínimo estendido. É o recurso mais
característico de um banco de produção como o Qdrant. Python puro; o Qdrant fica como referência.

## Busca com filtro por metadados

Cada documento tem um metadado (a disciplina). A busca pode ser restrita a uma disciplina,
combinando similaridade semântica com um filtro exato.

In [ ]:
import re
import math
from collections import Counter


def tok(texto):
    return re.findall(r"\w+", texto.lower())


itens = [
    {"texto": "A derivada mede a taxa de variação de uma função.", "disciplina": "calculo"},
    {"texto": "A integral acumula valores ao longo de um intervalo.", "disciplina": "calculo"},
    {"texto": "Uma matriz organiza números em linhas e colunas.", "disciplina": "algebra"},
    {"texto": "O determinante indica se uma matriz é invertível.", "disciplina": "algebra"},
]

N = len(itens)
df = Counter()
for it in itens:
    for w in set(tok(it["texto"])):
        df[w] += 1
idf = {w: math.log(N / f) for w, f in df.items()}


def vetor(texto):
    tf = Counter(tok(texto))
    return {w: tf[w] * idf.get(w, 0.0) for w in tf}


def cos(a, b):
    p = sum(a[w] * b.get(w, 0.0) for w in a)
    na = math.sqrt(sum(v * v for v in a.values()))
    nb = math.sqrt(sum(v * v for v in b.values()))
    return p / (na * nb) if na and nb else 0.0


def buscar(pergunta, filtro_disciplina=None, k=2):
    q = vetor(pergunta)
    candidatos = [it for it in itens
                  if filtro_disciplina is None or it["disciplina"] == filtro_disciplina]
    ranking = sorted(((cos(q, vetor(it["texto"])), it) for it in candidatos),
                     reverse=True, key=lambda par: par[0])
    return [(round(s, 3), it["texto"]) for s, it in ranking[:k]]


print("Sem filtro:")
for s, t in buscar("o que é uma matriz?"):
    print("  ", s, t)

print("\nFiltrando só álgebra:")
for s, t in buscar("o que é uma matriz?", filtro_disciplina="algebra"):
    print("  ", s, t)

A busca filtrada por álgebra ignora os documentos de cálculo, retornando só o que
interessa daquela disciplina. Esse recurso é nativo e eficiente no Qdrant. A lógica de RAG não
muda, só o componente de armazenamento fica mais poderoso.

Para o projeto que fecha o módulo, monte o assistente educacional completo em
`projects/m09-rag-assistant/`, com indexação, banco vetorial, busca com citação, tratamento da
ausência e filtro por disciplina.